This notebook is for preprocessing the data and getting it ready to be clustered. This is done by exploring the data, getting rid of outliers, 
and converting data to binary to only check for presence of a feature. I chose to use a variance threshhold for some feature reduction by only 
getting rid of features where all values were identical. If a feature is just the exact same (present or not present) for every sample of data, it 
contriubutes nothing but noise. I chose not to use Principal Componenet Analysis (PCA) because it assumes the data is continuouse, not binary, causing 
potential issues due to assumptions. Multiple Correspondence Analysis (MCA) would also work for dimensionality reduction, but would likely cause a loss 
in information.

1. Load data

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import VarianceThreshold
import re
import json
from functools import reduce

In [2]:
%load_ext autoreload
%autoreload 2

import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from src.utils import parquetParser, val_count, data_simplification, read_json

In [3]:
print(os.getcwd())

/Users/kennyhong/repos/StraboSpot/notebooks


In [4]:

df = parquetParser.parquet_parser(f"../data")

Running...
1
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1457251 entries, 0 to 1457250
Data columns (total 475 columns):
 #    Column                                                                Dtype 
---   ------                                                                ----- 
 0    properties_images                                                     object
 1    properties_date                                                       object
 2    properties_viewed_timestamp                                           object
 3    properties_notes                                                      object
 4    properties_orientation_data                                           object
 5    properties_modified_timestamp                                         object
 6    properties_symbology_circleColor                                      object
 7    properties_name                                                       object
 8    properties_notesTimestamp            

In [5]:
df.head()

,properties_images,properties_date,properties_viewed_timestamp,properties_notes,properties_orientation_data,properties_modified_timestamp,properties_symbology_circleColor,properties_name,properties_notesTimestamp,properties_time,...,properties_sed_structures_addl_mud_silt_struct,properties_orientation_unix_timestamp,properties_metamorphic,properties_sed_structures_cement_texture,properties_sed_structures_other_cement_texture,properties_sed_structures_other_addl_mud_silt_struct,properties_sed_structures_other_addl_sandstone_struct,properties_sed_structures_diversity,properties_sed_structures_behavioral_grouping,properties_sed_structures_list_of_specific_types
0,"[{""width"": 2224, ""id"": 16931574683185, ""modifi...",2023-08-27T17:25:12.000Z,1693157112029,Thick sequence of bedded chert. No sand!,"[{""id"": ""65a043ba-4177-44b3-8c8a-d4093bc4bddf""...",1693157726080,transparent,Bedded chert,Sun Aug 27 2023 10:25:41 GMT-0700 (Pacific Day...,2023-08-27T17:25:12.000Z,...,None,None,None,None,None,None,None,None,None,None
1,"[{""width"": 2448, ""id"": 16931711786083, ""modifi...",2023-08-27T20:43:07.000Z,1693168987963,"Badger creek unit includes chert argillite, li...","[{""id"": ""54169234-7ea4-463f-a6d4-a7a3115cd93e""...",1693171184906,transparent,23BCU5,Sun Aug 27 2023 14:13:29 GMT-0700 (Pacific Day...,2023-08-27T20:43:07.000Z,...,None,None,None,None,None,None,None,None,None,None
2,"[{""width"": 2224, ""id"": 16931765219731, ""modifi...",2023-08-27T22:28:21.000Z,1693175301286,Gray finely laminated possibly micrite inter l...,"[{""id"": ""9f95e7ae-e050-4606-8504-df0801870db9""...",1693176677608,transparent,23BCU6,Sun Aug 27 2023 15:51:17 GMT-0700 (Pacific Day...,2023-08-27T22:28:21.000Z,...,None,None,None,None,None,None,None,None,None,None
3,"[{""width"": 2448, ""id"": 16930789719922, ""modifi...",2023-08-26T18:31:54.000Z,1693074714285,Chert pebble breccia grades into fine sand. Si...,"[{""id"": ""b25ee609-01a4-4476-ba76-800c3151292d""...",1693089998466,transparent,23ERA4,Sat Aug 26 2023 12:03:10 GMT-0700 (Pacific Day...,2023-08-26T18:31:54.000Z,...,None,None,None,None,None,None,None,None,None,None
4,"[{""width"": 2224, ""id"": 16930050077699, ""modifi...",2023-08-25T21:56:07.000Z,1693000567149,Elk horn ridge argillite along road south of B...,None,1693009790925,transparent,23ERA1,Fri Aug 25 2023 16:34:19 GMT-0700 (Pacific Day...,2023-08-25T21:56:07.000Z,...,None,None,None,None,None,None,None,None,None,None


In [6]:
df.tail()

,properties_images,properties_date,properties_viewed_timestamp,properties_notes,properties_orientation_data,properties_modified_timestamp,properties_symbology_circleColor,properties_name,properties_notesTimestamp,properties_time,...,properties_sed_structures_addl_mud_silt_struct,properties_orientation_unix_timestamp,properties_metamorphic,properties_sed_structures_cement_texture,properties_sed_structures_other_cement_texture,properties_sed_structures_other_addl_mud_silt_struct,properties_sed_structures_other_addl_sandstone_struct,properties_sed_structures_diversity,properties_sed_structures_behavioral_grouping,properties_sed_structures_list_of_specific_types
1457246,None,2024-08-26T17:20:17+00:00,None,None,None,1724692817071,None,10458,None,2024-08-26T17:20:17+00:00,...,None,None,None,None,None,None,None,None,None,None
1457247,None,2024-08-26T17:18:00+00:00,None,None,None,1724692680599,None,7278,None,2024-08-26T17:18:00+00:00,...,None,None,None,None,None,None,None,None,None,None
1457248,None,2024-08-26T17:18:56+00:00,None,None,None,1724692736255,None,8584,None,2024-08-26T17:18:56+00:00,...,None,None,None,None,None,None,None,None,None,None
1457249,None,2024-08-26T17:17:39+00:00,None,None,None,1724692659000,None,6780,None,2024-08-26T17:17:39+00:00,...,None,None,None,None,None,None,None,None,None,None
1457250,None,2024-08-26T17:14:32+00:00,None,None,None,1724692472047,None,2328,None,2024-08-26T17:14:32+00:00,...,None,None,None,None,None,None,None,None,None,None


Some features may be unnecessary in the in analysis, especially features that pertain more to the use of the app and its features like line and circle colors for example. It is also a very necessary step to try and reduce the cardinality of the data when possible so as to reduce the noise later on and improve processing time.

In [7]:
# columns_to_drop = []
# for col in df.columns:
#     if col == "properties_orientation_modified_timestamp": # possibly indicative of purpose
#         continue 
#     elif col == "project_id":
#         continue
    
#     if re.search("^properties_symbology", col):
#         columns_to_drop.append(col)
#     if re.search("_id$", col):
#         columns_to_drop.append(col)
#     if re.search("_timestamp$", col):
#         columns_to_drop.append(col)
        
# columns_to_drop.append("properties_gps_accuracy")
# columns_to_drop.append("properties_altitude")
# columns_to_drop.append("properties_notesTimestamp")

# df = df.drop(columns= columns_to_drop)

# print("Columns Dropped:")
# for col in columns_to_drop:
#     print(f"\t{col}")



In [8]:
cols_to_drop = [
    "properties_viewed_timestamp",
	"properties_modified_timestamp",
	"properties_symbology_circleColor",
	"properties_id",
	"properties_samples_id",
	"properties_symbology_lineColor",
	"properties_symbology_lineWidth",
	"properties_symbology_lineDasharray",
	"properties_sed_strat_section_strat_section_id",
	"properties_strat_section_id",
	"properties_symbology_fillColor",
	"properties_orientation_id",
	"properties_custom_fields_osm_id",
	"properties_custom_fields_id",
	"properties_orientation_unix_timestamp",
	"properties_gps_accuracy",
	"properties_altitude",
	"properties_notesTimestamp",
]

df = df.drop(columns = cols_to_drop)

In [9]:
#getting rid of duplicates

df.drop_duplicates(inplace = True)
df.shape

(1411385, 457)

In [10]:
# drop the "properties_" part of each features name for better readability
df.columns = df.columns.str.replace('properties_', '', regex=False)
for col in df.columns:
    print(col)

images
date
notes
orientation_data
name
time
self
geometry_type
geometry_coordinates
type
account_email
project_id
dataset_name
samples_sample_id_name
samples_inplaceness_of_sample
samples_sample_notes
samples_oriented_sample
samples_main_sampling_purpose
samples_sample_size
altitude_accuracy
lng
image_basemap
lat
samples_sample_description
samples_material_type
samples_degree_of_weathering
samples_label
samples_sample_orientation_notes
_3d_structures
trace_trace_feature
trace_trace_quality
trace_trace_type
trace_contact_type
trace_intrusive_contact_type
trace_metamorphic_contact_type
trace_trace_character
samples_sample_type
spot_radius
images_notes
sed_strat_section_column_profile
sed_strat_section_column_y_axis_units
sed_strat_section_purpose
sed_strat_section_scale_of_interest
sed_strat_section_obs_interval_bed_obs_scale
sed_strat_section_section_type
sed_strat_section_basin
sed_strat_section_age
sed_character
sed_interval_interval_thickness
sed_interval_thickness_units
surface_fea

2. Explore features and properly split/encode them

In [11]:
val_count.val_count(df)

0
images
[{"lng": -105.28238207112, "width": 1536, "id": 14974604412492, "lat": 38.406684324181, "height": 1536, "image_type": "photo", "annotated": false, "self": "https://strabospot.org/db/image/14974604412492"}]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  

In [12]:
# images seems to contain a dictionary for that contain metadata per picture. 
# I will split the "images" feature up to properly reflect this. I will exclude the "widgth", "height", "id", "timestamp", and "self" 
df[["images"]].value_counts().tail(5)

images                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           
[{"width": 2448, "id": 16261273630552, "modified_timestamp": "1631562787942", "height": 1836, "image_type": "photo", "annotated": false, "self": "https://strabospot.org/db/image/16261273630552"}, {"width": 2448, "id": 16261273570267, "modified_timestamp": "1631562785810", "height": 1836, "image_type": "photo", "annotated": false, "self": "https://strabospot.org/db/image/1

In [13]:
json_output_cols = [
    col
    for col in df.columns
    if df[col].dropna().astype(str).str.startswith(("{", "[")).any()
]

In [14]:
print(json_output_cols)

['images', 'notes', 'orientation_data', 'name', 'geometry_coordinates', 'lng', '_3d_structures', 'sed_strat_section_purpose', 'sed_strat_section_scale_of_interest', 'sed_lithologies', 'sed_bedding_beds', 'geometry_geometries', 'sed_strat_section_images', 'other_features', 'tephra', 'sed_fossils', 'pet_rock_type', 'pet_igneous_rock_class', 'pet_volcanic_rock_type', 'pet_occurence_volcanic', 'pet_metamorphic_rock_type', 'sed_structures', 'sed_interpretations', 'pet_texture_volcanic', 'pet_alteration_volcanic', 'pet_plutonic_rock_type', 'pet_occurence_plutonic', 'pet_texture_plutonic', 'pet_minerals', 'pet_facies', 'pet_zone', 'fabrics', 'pet_alteration_plutonic', 'pet_igneous', 'inferences_relationships', 'pet_metamorphic', 'pet_metamorphic_rock_type_less_com', 'sed_diagenesis', 'nesting', 'pet_reactions', 'data_tables', 'orientation_movement_justification', 'orientation_directional_indicators', 'orientation_associated_orientation', 'sed_interpretations_clastic', 'sed_interpretations_sha

In [15]:
len(json_output_cols)

116

In [16]:
# import json
# import pandas as pd

# KEYS_TO_DROP = {"width", "height", "id", "modified_timestamp", "self"}
# BATCH_SIZE = 10_000 
# COLS_TO_FLATTEN = json_output_cols # Add all target columns here

# def flatten_json_field(raw, col_prefix):
#     """Turn one JSON string into a flat dict prefixed by col_prefix."""
#     try:
#         items = json.loads(raw)
#     except (TypeError, json.JSONDecodeError):
#         return {}

#     if not isinstance(items, list):
#         return {}

#     flat = {}
#     for k, item in enumerate(items):
#         if k > 4:  # Keep only first 5
#             break
        
#         suffix = "" if k == 0 else f"_{k + 1}"
#         if isinstance(item, dict):
#             for key, value in item.items():
#                 if key not in KEYS_TO_DROP:
#                     flat[f"{col_prefix}_{key}{suffix}"] = value
#     return flat

# # --- BATCH PROCESSING ---
# batches = []

# for start in range(0, len(df), BATCH_SIZE):
#     chunk = df.iloc[start:start + BATCH_SIZE]
    
#     # List to store the merged flattened row dicts for this chunk
#     chunk_rows = []
    
#     # Iterate row by row using itertuples or zipped values
#     for row in chunk[COLS_TO_FLATTEN].itertuples(index=False):
#         row_dict = {}
#         # Iterate across every column specified in COLS_TO_FLATTEN
#         for col_name, raw_val in zip(COLS_TO_FLATTEN, row):
#             flat_col = flatten_json_field(raw_val, col_prefix=col_name)
#             row_dict.update(flat_col)  # Combine results into a single row dictionary
            
#         chunk_rows.append(row_dict)
        
#     batches.append(pd.DataFrame(chunk_rows, index=chunk.index))

# # Combine all batch DataFrames and join back to original DataFrame
# flattened_df = pd.concat(batches)
# df = df.join(flattened_df, rsuffix="_new")

In [17]:
import gc

try:  # optional, ~3-5x faster than stdlib json and lighter on allocations
    import orjson
    _loads = orjson.loads
    _DecodeError = orjson.JSONDecodeError
except ImportError:
    _loads = json.loads
    _DecodeError = json.JSONDecodeError

KEYS_TO_DROP = frozenset({"width", "height", "id", "modified_timestamp", "self"})
MAX_ITEMS = 5
BATCH_SIZE = 10_000
COLS_TO_FLATTEN = json_output_cols # your existing list of target columns

# Cache of output column names so we don't rebuild millions of f-strings
_name_cache = {}


def _name(col, key, k):
    ck = (col, key, k)
    n = _name_cache.get(ck)
    if n is None:
        n = f"{col}_{key}" + ("" if k == 0 else f"_{k + 1}")
        _name_cache[ck] = n
    return n


def flatten_cell(col, raw, out):
    """Parse one JSON string and write flattened values straight into `out`."""
    if not raw or raw == "[]":
        return
    try:
        items = _loads(raw)
    except (TypeError, ValueError, _DecodeError):
        return
    if not isinstance(items, list):
        return
    for k, item in enumerate(items[:MAX_ITEMS]):
        if isinstance(item, dict):
            for key, value in item.items():
                if key in KEYS_TO_DROP:
                    continue
                # nested containers are expensive as Python objects; keep as compact JSON text
                if isinstance(value, (dict, list)):
                    value = json.dumps(value, separators=(",", ":"))
                out[_name(col, key, k)] = value


def flatten_column(series):
    """Flatten one raw JSON column into a compact (pyarrow-backed) DataFrame."""
    col = series.name
    parts = []
    for start in range(0, len(series), BATCH_SIZE):
        raw_values = series.iloc[start:start + BATCH_SIZE].tolist()
        rows = []
        for raw in raw_values:
            row = {}
            flatten_cell(col, raw, row)
            rows.append(row)
        part = pd.DataFrame(rows).convert_dtypes(dtype_backend="pyarrow")
        parts.append(part)
        del raw_values, rows, part
    # concat fills columns missing from some batches with <NA>
    out = pd.concat(parts, ignore_index=True)
    del parts
    out.index = series.index
    return out


# --- MAIN: one source column at a time, dropping the raw column immediately ---
flattened = []
for col in COLS_TO_FLATTEN:
    out = flatten_column(df[col])
    df.drop(columns=col, inplace=True)  # free the raw JSON strings right away
    flattened.append(out)
    del out
    gc.collect()

# Single concat at the end (raw JSON columns are already gone, so this is much smaller)
df = pd.concat([df, *flattened], axis=1)
del flattened, _name_cache
gc.collect()

# Optional: shrink the remaining original columns too
# for c in df.select_dtypes("object").columns:
#     if df[c].nunique(dropna=True) / max(len(df), 1) < 0.5:
#         df[c] = df[c].astype("category")
# df = df.convert_dtypes(dtype_backend="pyarrow")

print(f"{df.shape=}, {df.memory_usage(deep=True).sum() / 1e6:,.1f} MB")

KeyboardInterrupt: 

In [ ]:
print(df.filter(like = "images").head().to_markdown())

|    | images                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
df["images_annotated"].value_counts()

images_annotated
False    206321
True       7522
Name: count, dtype: int64

In [34]:
# uses variance threshold for feature reduction, removes features where all values are identical

selector = VarianceThreshold(threshold=0) #removes all features with low variance in 100% of samples  (you do (1 - percentage of same values) * percentage of vaues that are the same)
selector.fit_transform(df)

cols_idxs = selector.get_support(indices=True)
df = df.iloc[:,cols_idxs]

print(df.shape)
# val_count(df)


(1411385, 433)


In [35]:
for columns in df.columns:
    print(columns)

project_id
properties_images
properties_notes
properties_orientation_data
properties_name
geometry_type
geometry_coordinates
dataset_name
properties_samples_sample_id_name
properties_samples_inplaceness_of_sample
properties_samples_sample_notes
properties_samples_oriented_sample
properties_samples_main_sampling_purpose
properties_samples_sample_size
properties_altitude_accuracy
properties_lng
properties_image_basemap
properties_lat
properties_samples_sample_description
properties_samples_material_type
properties_samples_degree_of_weathering
properties_samples_label
properties_samples_sample_orientation_notes
properties__3d_structures
properties_trace_trace_feature
properties_trace_trace_quality
properties_trace_trace_type
properties_trace_contact_type
properties_trace_intrusive_contact_type
properties_trace_metamorphic_contact_type
properties_trace_trace_character
properties_samples_sample_type
properties_spot_radius
properties_images_notes
properties_sed_strat_section_column_profile
p

In [36]:
def find_purpose(spot, purpose_features, purpose_threshold = 2):
    # print(spot)
    features_sum = 0
    not_features_sum = 0
    # not_gm_features = [f for f in spot.index if f not in gm_features]
        
    for feature in spot.index:
        if spot[feature] == 1 and feature in purpose_features:
            features_sum += 1
        elif spot[feature] == 1 and feature not in purpose_features:
            not_features_sum += 1
            
    purpose_presence = features_sum
    not_purpose_presence = not_features_sum
    
    # if no features that are not a part of the purpose are present, assume spot is the purpose
    if not_purpose_presence == 0:
        return True
    
    if purpose_presence / not_purpose_presence >= purpose_threshold: 
        return True
    else:
        return False

In [37]:
df_copy = df.copy()

In [38]:
gm_features = ["properties_rock_unit_unit_label_abbreviation",
               "properties_images_notes",
               "properties_images",
               "properties_notes",
               "properties_pet_rock_type",
               "geometry_coordinates",
               "properties_images"
               "properties_orientation_data", 
               "properties_trace_trace_feature", 
               "properties_trace_trace_quality", 
               "properties_trace_trace_type", 
               "properties_trace_contact_type", 
               "properties_other_features"
               ]

site_safety_features = ["properties_site_safety_site_summary_author",
                        "properties_site_safety_field_stop_designation",
                        "properties_site_safety_longitude",
                        "properties_site_safety_latitude"
                        ]

strat_features = [ "properties_sed_strat_section_column_profile",
                  "properties_sed_strat_section_column_y_axis_units",
                  "properties_sed_strat_section_purpose",
                  "properties_sed_strat_section_scale_of_interest",
                  "properties_sed_strat_section_obs_interval_bed_obs_scale",
                  "properties_sed_strat_section_section_type",
                  "properties_sed_strat_section_basin",
                  "properties_sed_strat_section_age",
                  "properties_sed_strat_section_total_core_length",
                  "properties_sed_strat_section_images",
                  "properties_sed_strat_section_section_well_name",
                  "properties_sed_strat_section_location_locality",
                  "properties_sed_strat_section_dates_of_work",
                  "properties_sed_strat_section_strat_section_notes",
                  "properties_sed_strat_section_display_lithology_patterns",
                  "properties_sed_strat_section_project_description",
                  "properties_sed_strat_section_how_is_section_georeferenced",
                  "properties_sed_strat_section_misc_labels",
                  "properties_sed_strat_section_what_core_repository",
                  "properties_sed_strat_section_type_of_corer",
                  "properties_sed_strat_section_depth_from_surface_to_start_of",
                  "properties_sed_strat_section_other_purpose",
                  "properties_sed_strat_section_label",
                  "properties_sed_strat_section_other_scale_of_interest"
                  ]




In [39]:
df.shape

(1411385, 433)

In [16]:
# df["is_general_mapping"] = df.apply(lambda x: find_purpose(x,gm_features, purpose_threshold= 1.5), axis = 1)


In [17]:
val_count(df.iloc[:, 440:])

(1416767, 0)


In [18]:
# gm_ids = df[df["is_general_mapping"] == True]["project_id"]

# spots_to_drop = df["project_id"].isin(gm_ids)

# df = df[~spots_to_drop]

In [19]:
df.shape

(1416767, 436)

In [20]:
# df.drop(df[df["is_general_mapping"] == True].index, inplace= True)
# df = df.drop("project_id", axis = 1)
# df = df.drop("is_general_mapping", axis= 1)
# print(df.shape)

In [21]:
df.to_csv("../data/parquets/processed_parquet.csv", index = False)
